# PCL Detection — SemEval 2022 Task 4 (Subtask 1)

**Task**: Binary classification — does a paragraph contain Patronizing/Condescending Language (PCL)?

**Model**: `microsoft/deberta-v3-base` — stronger encoder than the RoBERTa-base baseline

**Approach**:
1. Class-weighted `BCEWithLogitsLoss` (pos_weight ≈ 9.55) to handle 1:9.5 class imbalance
2. Multi-task learning: auxiliary 7-category PCL loss forces richer representations
3. Threshold tuning on dev set to maximise F1

**Baseline**: RoBERTa-base, standard loss, t=0.5 → F1 = 0.48 (dev), 0.49 (test)

In [31]:
# Install required packages — run this cell once, then restart the kernel
# !pip install transformers>=4.35.0 sentencepiece protobuf torch scikit-learn pandas numpy tqdm

# Packages needed and why:
#   transformers  — AutoTokenizer, AutoModelForSequenceClassification
#   sentencepiece — DeBERTa-v3 uses a SentencePiece tokenizer (.spm.model)
#   protobuf      — required by sentencepiece to parse the binary .spm.model file
#
# NOTE: tiktoken is NOT needed — it is for OpenAI/GPT models only

In [32]:
import os, ast, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import (
    f1_score, precision_score, recall_score, classification_report
)
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    bf16_ok = torch.cuda.is_bf16_supported()
    print(f"BF16   : {'supported' if bf16_ok else 'NOT supported — will use FP32'}")

Device : cuda
GPU    : NVIDIA GeForce RTX 5070
VRAM   : 12.8 GB
BF16   : supported


In [33]:
# ── Configuration ────────────────────────────────────────────────────────────
import os

CFG = {
    # Model
    'model_name':    'microsoft/deberta-v3-base',
    # Data files
    'main_data':     'dontpatronizeme_pcl.tsv',
    'train_ids':     'train_semeval_parids-labels.csv',
    'dev_ids':       'dev_semeval_parids-labels.csv',
    'test_data':     'task4_test.tsv',
    # Tokenizer
    'max_length':    128,
    # Training
    'batch_size':    16,
    'grad_accum':    2,          # effective batch = 32
    'lr':            2e-5,
    'weight_decay':  0.01,
    'warmup_ratio':  0.10,
    'num_epochs':    8,
    'aux_weight':    0.3,        # weight for auxiliary 7-category loss
    'seed':          42,
    # Output files (BestModel/ folder — required by spec Exercise 4 & 5.1)
    'best_model':    'BestModel/best_model.pt',
    'dev_preds':     'BestModel/dev.txt',
    'test_preds':    'BestModel/test.txt',
}

PCL_CATEGORIES = [
    'Unbalanced power relations',
    'Shallow solution',
    'Presupposition',
    'Authority voice',
    'Metaphor',
    'Compassion',
    'The poorer the merrier',
]

os.makedirs('BestModel', exist_ok=True)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(CFG['seed'])
print("Config ready.  BestModel/ folder created.")

Config ready.  BestModel/ folder created.


## 1. Data Loading

In [34]:
# ── Load main dataset (dontpatronizeme_pcl.tsv) ──────────────────────────────
# Uses a line-by-line parser that safely skips the 4-row disclaimer header
# and any malformed rows.

def load_main_data(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.rstrip('\n').split('\t', 5)
            if len(parts) == 6 and parts[0].strip().isdigit():
                rows.append({
                    'par_id':    int(parts[0]),
                    'keyword':   parts[2],
                    'country':   parts[3],
                    'text':      parts[4].strip(),
                    'label_raw': int(parts[5]),
                })
    df = pd.DataFrame(rows)
    # Task 3 binary label: PCL if >= 2 annotators flagged it
    df['label_binary'] = (df['label_raw'] >= 2).astype(int)
    return df


def load_split_ids(path):
    """Load par_ids and their 7-element Task-4 multi-label vectors."""
    df = pd.read_csv(path)
    df['par_id'] = df['par_id'].astype(int)
    df['label_list'] = df['label'].apply(ast.literal_eval)
    return df[['par_id', 'label_list']]


def load_test_data(path):
    """Load test set — par_ids start with 't_', no label column, no disclaimer."""
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.rstrip('\n').split('\t')
            if len(parts) >= 5 and parts[0].strip().startswith('t_'):
                rows.append({
                    'par_id':  parts[0].strip(),
                    'keyword': parts[2] if len(parts) > 2 else '',
                    'country': parts[3] if len(parts) > 3 else '',
                    'text':    parts[4].strip() if len(parts) > 4 else '',
                })
    df = pd.DataFrame(rows)
    df = df[df['text'].str.len() > 0].reset_index(drop=True)
    return df


# Load everything
df_main      = load_main_data(CFG['main_data'])
df_train_ids = load_split_ids(CFG['train_ids'])
df_dev_ids   = load_split_ids(CFG['dev_ids'])
df_test      = load_test_data(CFG['test_data'])

print(f"Main dataset : {len(df_main):>6,} rows")
print(f"Train IDs    : {len(df_train_ids):>6,} rows")
print(f"Dev IDs      : {len(df_dev_ids):>6,} rows")
print(f"Test set     : {len(df_test):>6,} rows")

Main dataset : 10,469 rows
Train IDs    :  8,375 rows
Dev IDs      :  2,094 rows
Test set     :  3,832 rows


In [35]:
# ── Merge labels into train / dev splits ─────────────────────────────────────

df_train = df_main[df_main['par_id'].isin(df_train_ids['par_id'])].copy()
df_dev   = df_main[df_main['par_id'].isin(df_dev_ids['par_id'])].copy()

df_train = df_train.merge(df_train_ids, on='par_id', how='left').reset_index(drop=True)
df_dev   = df_dev.merge(df_dev_ids,   on='par_id', how='left').reset_index(drop=True)

# Fill Task-4 labels with all-zeros for samples not in the label CSV
_zero7 = lambda x: x if isinstance(x, list) else [0] * 7
df_train['label_list'] = df_train['label_list'].apply(_zero7)
df_dev['label_list']   = df_dev['label_list'].apply(_zero7)

print(f"Train : {len(df_train):,} samples")
print(f"Dev   : {len(df_dev):,} samples")

# Task 3 class breakdown
n_neg_tr = (df_train['label_binary'] == 0).sum()
n_pos_tr = (df_train['label_binary'] == 1).sum()
n_neg_dv = (df_dev['label_binary'] == 0).sum()
n_pos_dv = (df_dev['label_binary'] == 1).sum()
print(f"\nTask 3 — Train: No-PCL={n_neg_tr:,}  PCL={n_pos_tr:,}  ratio={n_neg_tr/n_pos_tr:.1f}:1")
print(f"Task 3 — Dev  : No-PCL={n_neg_dv:,}  PCL={n_pos_dv:,}")

Train : 8,375 samples
Dev   : 2,094 samples

Task 3 — Train: No-PCL=7,581  PCL=794  ratio=9.5:1
Task 3 — Dev  : No-PCL=1,895  PCL=199


In [36]:
# ── Data verification ─────────────────────────────────────────────────────────

# 1. No overlap between train and dev
overlap = set(df_train['par_id']) & set(df_dev['par_id'])
print(f"Train/dev par_id overlap (should be 0): {len(overlap)}")

# 2. Task-4 label array stats
train_arr = np.array(df_train['label_list'].tolist(), dtype=np.float32)
dev_arr   = np.array(df_dev['label_list'].tolist(),   dtype=np.float32)

print(f"\nTask 4 per-category positive counts (train) — {len(train_arr):,} samples total:")
for i, cat in enumerate(PCL_CATEGORIES):
    cnt = int(train_arr[:, i].sum())
    pct = train_arr[:, i].mean() * 100
    print(f"  [{i}] {cat:<35s} : {cnt:>4} ({pct:.1f}%)")

# 3. Quick text sample
print("\nSample row:")
print(df_train[df_train['label_binary'] == 1].iloc[0][['text', 'label_binary', 'label_list']].to_string())

Train/dev par_id overlap (should be 0): 0

Task 4 per-category positive counts (train) — 8,375 samples total:
  [0] Unbalanced power relations          :  574 (6.9%)
  [1] Shallow solution                    :  160 (1.9%)
  [2] Presupposition                      :  162 (1.9%)
  [3] Authority voice                     :  192 (2.3%)
  [4] Metaphor                            :  145 (1.7%)
  [5] Compassion                          :  363 (4.3%)
  [6] The poorer the merrier              :   29 (0.3%)

Sample row:
text            Arshad said that besides learning many new asp...
label_binary                                                    1
label_list                                  [1, 0, 0, 0, 0, 0, 0]


## 2. Dataset Class & Training Utilities

In [37]:
# ── Tokenizer ────────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(CFG['model_name'])
print(f"Tokenizer loaded: {CFG['model_name']}")


# ── Dataset class ─────────────────────────────────────────────────────────────
class PCLDataset(Dataset):
    """
    Binary PCL dataset.  Optionally carries 7-category aux_labels for
    multi-task training (train/dev only — test set has no category labels).
    """
    def __init__(self, texts, labels, tokenizer, max_length, aux_labels=None):
        self.texts      = [str(t) for t in texts]
        self.labels     = labels
        self.tokenizer  = tokenizer
        self.max_length = max_length
        self.aux_labels = aux_labels   # list of 7-element lists, or None

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item['labels'] = torch.tensor(float(self.labels[idx]), dtype=torch.float)
        if self.aux_labels is not None:
            item['aux_labels'] = torch.tensor(self.aux_labels[idx], dtype=torch.float)
        return item

Tokenizer loaded: microsoft/deberta-v3-base


In [38]:
# ── Training / evaluation / prediction helpers ───────────────────────────────
# Pure FP32: DeBERTa-v3 disentangled attention + StableDropout overflow in
# BF16/FP16 on Blackwell (RTX 50-series). FP32 is stable and fast enough here.
#
# Multi-task setup:
#   model has num_labels=8  →  logits shape (batch, 8)
#   logits[:, 0]   = binary PCL logit   (primary task)
#   logits[:, 1:]  = 7-category logits  (auxiliary task, weight = aux_weight)

def run_epoch(model, loader, optimizer, scheduler, criterion,
              grad_accum, aux_criterion=None, aux_weight=0.3):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()

    pbar = tqdm(enumerate(loader), total=len(loader), desc='Train', leave=True)
    for step, batch in pbar:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        extra = {}
        if 'token_type_ids' in batch:
            extra['token_type_ids'] = batch['token_type_ids'].to(device)

        logits = model(input_ids=input_ids, attention_mask=attention_mask, **extra).logits

        # Primary binary loss
        loss = criterion(logits[:, 0], labels)

        # Auxiliary 7-category loss (train/dev have labels; test does not)
        if aux_criterion is not None and 'aux_labels' in batch:
            aux_labels = batch['aux_labels'].to(device)
            loss = loss + aux_weight * aux_criterion(logits[:, 1:], aux_labels)

        loss = loss / grad_accum
        loss.backward()

        if (step + 1) % grad_accum == 0 or (step + 1) == len(loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * grad_accum
        pbar.set_postfix({'loss': f'{total_loss / (step + 1):.4f}'})

    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, threshold=0.5):
    model.eval()
    all_logits, all_labels = [], []

    for batch in tqdm(loader, desc='Eval', leave=False):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        extra = {}
        if 'token_type_ids' in batch:
            extra['token_type_ids'] = batch['token_type_ids'].to(device)

        logits = model(input_ids=input_ids, attention_mask=attention_mask, **extra).logits
        all_logits.append(logits[:, 0].cpu().float().numpy())   # binary logit only
        all_labels.append(batch['labels'].numpy())

    all_logits = np.concatenate(all_logits, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    probs = 1.0 / (1.0 + np.exp(-all_logits))
    preds = (probs >= threshold).astype(int)
    f1    = f1_score(all_labels.astype(int), preds, zero_division=0)

    return f1, probs, all_labels


def tune_threshold(probs, labels):
    best_t, best_f = 0.5, 0.0
    for t in np.linspace(0.05, 0.95, 91):
        p = (probs >= t).astype(int)
        f = f1_score(labels.astype(int), p, zero_division=0)
        if f > best_f:
            best_f, best_t = f, t
    print(f"Optimal threshold: {best_t:.3f}  →  F1 = {best_f:.4f}")
    return float(best_t)


@torch.no_grad()
def predict(model, loader, threshold):
    model.eval()
    all_probs = []

    for batch in tqdm(loader, desc='Predict', leave=False):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        extra = {}
        if 'token_type_ids' in batch:
            extra['token_type_ids'] = batch['token_type_ids'].to(device)

        logits = model(input_ids=input_ids, attention_mask=attention_mask, **extra).logits
        all_probs.append(torch.sigmoid(logits[:, 0]).cpu().float().numpy())

    all_probs = np.concatenate(all_probs, axis=0)
    preds     = (all_probs >= threshold).astype(int)
    return preds, all_probs


print("Utilities defined (pure FP32 · multi-task).")

Utilities defined (pure FP32 · multi-task).


---
## 3. Binary PCL Detection

**Baseline**: RoBERTa-base, standard CE loss, t=0.5 → **F1 = 0.48 (dev), 0.49 (test)**

**Our approach**:
- **DeBERTa-v3-base**: disentangled attention + ELECTRA-style pretraining — stronger than RoBERTa-base at the same ~86M parameter count
- **Class-weighted `BCEWithLogitsLoss`** with `pos_weight ≈ 9.55` — directly counteracts the 1:9.5 class imbalance; prevents the model collapsing to always predicting No-PCL
- **Multi-task auxiliary loss** (weight = 0.3): the model outputs 8 logits — logit[0] is the binary PCL prediction, logits[1–7] are the 7 PCL sub-categories provided in the dataset. The auxiliary signal forces the encoder to capture fine-grained PCL features, improving generalisation on the primary binary task
- **Threshold tuning**: grid search over [0.05, 0.95] on dev set selects the threshold maximising F1
- FP32 training, gradient accumulation (effective batch = 32), AdamW + linear warmup

In [39]:
# ── Data loaders ─────────────────────────────────────────────────────────────
# Train and dev get aux_labels (7 categories) for multi-task training.
# Test set has no category labels — aux_labels=None.

train_ds = PCLDataset(df_train['text'].tolist(),
                      df_train['label_binary'].tolist(),
                      tokenizer, CFG['max_length'],
                      aux_labels=df_train['label_list'].tolist())

dev_ds   = PCLDataset(df_dev['text'].tolist(),
                      df_dev['label_binary'].tolist(),
                      tokenizer, CFG['max_length'],
                      aux_labels=df_dev['label_list'].tolist())

test_ds  = PCLDataset(df_test['text'].tolist(),
                      [0] * len(df_test),
                      tokenizer, CFG['max_length'])   # no aux labels

train_dl = DataLoader(train_ds, batch_size=CFG['batch_size'],
                      shuffle=True,  num_workers=0, pin_memory=True)
dev_dl   = DataLoader(dev_ds,   batch_size=CFG['batch_size'] * 2,
                      shuffle=False, num_workers=0, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=CFG['batch_size'] * 2,
                      shuffle=False, num_workers=0)

# ── Loss functions ────────────────────────────────────────────────────────────
# Primary: class-weighted binary BCE
n_neg = int((df_train['label_binary'] == 0).sum())
n_pos = int((df_train['label_binary'] == 1).sum())
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float).to(device)
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
print(f"Binary pos_weight = {pos_weight.item():.2f}  (No-PCL={n_neg}, PCL={n_pos})")

# Auxiliary: per-label weights for 7 PCL categories
train_arr  = np.array(df_train['label_list'].tolist(), dtype=np.float32)
pos_counts = train_arr.sum(axis=0).clip(min=1)
neg_counts = len(train_arr) - pos_counts
pw_aux     = torch.tensor(neg_counts / pos_counts, dtype=torch.float).to(device)
aux_criterion = nn.BCEWithLogitsLoss(pos_weight=pw_aux)
print("Auxiliary criterion ready (7 categories):")
for cat, w in zip(PCL_CATEGORIES, pw_aux.cpu().numpy()):
    print(f"  {cat:<35s}: pos_weight={w:.1f}")

# ── Model ─────────────────────────────────────────────────────────────────────
# num_labels=8: logit[0] = binary PCL, logits[1:8] = 7 PCL sub-categories
set_seed(CFG['seed'])
model = AutoModelForSequenceClassification.from_pretrained(
    CFG['model_name'], num_labels=8, torch_dtype=torch.float32
).to(device)

total_steps  = (len(train_dl) // CFG['grad_accum']) * CFG['num_epochs']
warmup_steps = int(total_steps * CFG['warmup_ratio'])

optimizer = AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)
print(f"\nModel: {CFG['model_name']}  |  num_labels=8")
print(f"Total steps: {total_steps} | Warmup: {warmup_steps}")

Binary pos_weight = 9.55  (No-PCL=7581, PCL=794)
Auxiliary criterion ready (7 categories):
  Unbalanced power relations         : pos_weight=13.6
  Shallow solution                   : pos_weight=51.3
  Presupposition                     : pos_weight=50.7
  Authority voice                    : pos_weight=42.6
  Metaphor                           : pos_weight=56.8
  Compassion                         : pos_weight=22.1
  The poorer the merrier             : pos_weight=287.8


Loading weights: 100%|██████████| 198/198 [00:00<00:00, 858.12it/s, Materializing param=deberta.encoder.rel_embeddings.weight]                     
DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
pooler.dense.weight                     | MIS


Model: microsoft/deberta-v3-base  |  num_labels=8
Total steps: 2096 | Warmup: 209


In [40]:
# ── Training loop ─────────────────────────────────────────────────────────────
best_f1     = 0.0
best_probs  = None
best_labels = None

for epoch in range(CFG['num_epochs']):
    print(f"\n── Epoch {epoch + 1} / {CFG['num_epochs']} ──")
    train_loss = run_epoch(
        model, train_dl, optimizer, scheduler, criterion,
        CFG['grad_accum'],
        aux_criterion=aux_criterion,
        aux_weight=CFG['aux_weight'],
    )
    f1, probs, labels = evaluate(model, dev_dl, threshold=0.5)
    print(f"   Train loss: {train_loss:.4f}  |  Dev F1 (t=0.5): {f1:.4f}")

    if f1 > best_f1:
        best_f1     = f1
        best_probs  = probs.copy()
        best_labels = labels.copy()
        torch.save(model.state_dict(), CFG['best_model'])
        print(f"   ✓ New best  (F1={best_f1:.4f}) — saved to {CFG['best_model']}")

print(f"\nBest dev F1 before threshold tuning: {best_f1:.4f}")


── Epoch 1 / 8 ──


Train: 100%|██████████| 524/524 [01:16<00:00,  6.84it/s, loss=1.6300]


   Train loss: 1.6300  |  Dev F1 (t=0.5): 0.3929
   ✓ New best  (F1=0.3929) — saved to BestModel/best_model.pt

── Epoch 2 / 8 ──


Train: 100%|██████████| 524/524 [01:16<00:00,  6.87it/s, loss=1.4519]


   Train loss: 1.4519  |  Dev F1 (t=0.5): 0.3636

── Epoch 3 / 8 ──


...

   Train loss: 0.9145  |  Dev F1 (t=0.5): 0.4988
   ✓ New best  (F1=0.4988) — saved to BestModel/best_model.pt

── Epoch 7 / 8 ──


Train: 100%|██████████| 524/524 [01:15<00:00,  6.96it/s, loss=0.8592]


   Train loss: 0.8592  |  Dev F1 (t=0.5): 0.4760

── Epoch 8 / 8 ──


Train: 100%|██████████| 524/524 [01:15<00:00,  6.91it/s, loss=0.7468]
                                                     

   Train loss: 0.7468  |  Dev F1 (t=0.5): 0.4902

Best dev F1 before threshold tuning: 0.4988


In [41]:
# ── Threshold tuning + final dev evaluation ───────────────────────────────────
model.load_state_dict(torch.load(CFG['best_model'], map_location=device))
_, best_probs, best_labels = evaluate(model, dev_dl, threshold=0.5)

print("Searching optimal threshold on dev set ...")
best_thresh = tune_threshold(best_probs, best_labels)

preds_dev = (best_probs >= best_thresh).astype(int)

print(f"\n{'=' * 55}")
print(f"  FINAL RESULTS  (threshold = {best_thresh:.3f})")
print(f"{'=' * 55}")
print(f"  Baseline (RoBERTa-base, t=0.5)   F1 = 0.4800")
final_f1 = f1_score(best_labels.astype(int), preds_dev)
print(f"  Our model (DeBERTa-v3-base)       F1 = {final_f1:.4f}")
print(f"  Improvement: +{final_f1 - 0.48:.4f}")
print()
print(classification_report(
    best_labels.astype(int), preds_dev,
    target_names=['No-PCL', 'PCL'], digits=4
))

# Save dev.txt — one prediction per line (required format, Exercise 5.1)
with open(CFG['dev_preds'], 'w') as f:
    for p in preds_dev:
        f.write(f"{int(p)}\n")
print(f"Saved → {CFG['dev_preds']}  ({len(preds_dev)} lines)")

Searching optimal threshold on dev set ...
Optimal threshold: 0.840  →  F1 = 0.5061

  FINAL RESULTS  (threshold = 0.840)
  Baseline (RoBERTa-base, t=0.5)   F1 = 0.4800
  Our model (DeBERTa-v3-base)       F1 = 0.5061
  Improvement: +0.0261

              precision    recall  f1-score   support

      No-PCL     0.9491    0.9446    0.9468      1895
         PCL     0.4952    0.5176    0.5061       199

    accuracy                         0.9040      2094
   macro avg     0.7221    0.7311    0.7265      2094
weighted avg     0.9060    0.9040    0.9050      2094

Saved → BestModel/dev.txt  (2094 lines)


In [42]:
# ── Test set predictions → test.txt ──────────────────────────────────────────
preds_test, probs_test = predict(model, test_dl, threshold=best_thresh)

# Save test.txt — one prediction per line (required format, Exercise 5.1)
with open(CFG['test_preds'], 'w') as f:
    for p in preds_test:
        f.write(f"{int(p)}\n")

print(f"Saved → {CFG['test_preds']}  ({len(preds_test)} lines)")
print(f"Predicted PCL: {preds_test.sum()} / {len(preds_test)}  ({preds_test.mean()*100:.1f}%)")
print(f"\nFirst 10 predictions: {preds_test[:10].tolist()}")
print("\nBestModel/ contents:")
for fn in sorted(os.listdir('BestModel')):
    print(f"  {fn}")

Saved → BestModel/test.txt  (3832 lines)
Predicted PCL: 338 / 3832  (8.8%)

First 10 predictions: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

BestModel/ contents:
  best_model.pt
  dev.txt
  test.txt


---
## 5. Summary

| Model | Dev F1 | Notes |
|-------|--------|-------|
| RoBERTa-base (baseline) | 0.48 | Standard CE loss, t=0.5 |
| **DeBERTa-v3-base (ours)** | 0.5061 | Weighted BCE + multi-task aux loss + threshold tuning |

### Key design choices

1. **DeBERTa-v3-base over RoBERTa-base**: Disentangled attention encodes positional and content information separately, giving stronger representations for nuanced language like PCL. ELECTRA-style pretraining provides richer token-level supervision. Same ~86M parameters — no extra inference cost.

2. **Class-weighted BCE loss**: The training set has a 9.55:1 No-PCL:PCL ratio. Setting `pos_weight=9.55` makes each positive example contribute as much to the loss as 9.55 negatives, preventing the model from collapsing to always predicting No-PCL.

3. **Multi-task auxiliary loss** (weight=0.3): The 7 PCL sub-categories (Unbalanced power relations, Shallow solution, Presupposition, Authority voice, Metaphor, Compassion, The poorer the merrier) are provided in the dataset and used as an auxiliary training signal. The model predicts all 8 outputs simultaneously; only the binary logit is used at inference. Auxiliary supervision forces the encoder to learn finer-grained PCL features.

4. **Threshold tuning**: Grid search over [0.05, 0.95] on the dev set selects the classification threshold that directly maximises F1 on the positive class — typically lower than 0.5 on imbalanced data.

### Output files (`BestModel/`)

| File | Contents |
|------|----------|
| `dev.txt` | 2,094 binary predictions (0/1), one per line — Exercise 5.1 |
| `test.txt` | 3,832 binary predictions (0/1), one per line — Exercise 5.1 |
| `best_model.pt` | Best checkpoint weights — Exercise 4 |
| `pcl_detection.ipynb` | Full training notebook — Exercise 4 |